# Language Model Syntactic Productivity Analysis

This notebook analyzes results from syntactic productivity experiments on language models.

## Overview

We evaluate how well language models capture productive grammar patterns in determiner-noun combinations, comparing:
- **Isolated utterances**: Single sentences without context
- **Discourse context**: Sentences with multi-turn conversation history

## Prerequisites

Results should be generated first by running:
```bash
python lm_text_generation_exp.py both --models model_configs.json
```

In [1]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Circle
import seaborn as sns
from scipy.stats import ttest_rel, ttest_1samp, pearsonr
import os
import json
from pathlib import Path

# Import utilities\n
import cac_utils as lmtu
from cac_utils import load_overlap_results, load_tpr_results, generate_model_summaries

# Styling
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

/Users/hjvm/anaconda3/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: mps


In [2]:
# CONSTANTS — paper parameters, model selection, and output directory.
SPEAKERS = ["child", "mother"]
UNIV_BIAS = 0.82                # Average determiner bias as calculated in aggregate on COCA.
MAJORITY_DET_BASELINE = 0.535   # Baseline accuracy for always predicting most frequent determiner in Manchester corpus.

PASS_ALPHA = 0.05
TARGET_N_SELECTED = 8
# Fixed model set for main paper table and figures (in display order).
MAIN_MODEL_SPECS = [
    {"model": "ltg-bert-bnc", "arch": "MLM"},           # Pass Both
    {"model": "roberta-base", "arch": "MLM"},            # Pass Both
    {"model": "t5-base", "arch": "S2S"},                 # Pass TPR only (functional)
    {"model": "babylm-baseline-100m-gpt-bert-causal-focus", "arch": "AR"},  # Pass DxN only, high acc
    {"model": "gpt-bert-babylm-small", "arch": "MLM"},   # Pass DxN only, lower acc
    {"model": "opt-125m", "arch": "AR"},                 # Pass Neither, high acc
    {"model": "gpt2", "arch": "AR"},                     # Pass Neither, canonical baseline
    {"model": "roberta-base-strict-2023", "arch": "MLM"},# Pass Neither, low acc
]

# Explicit overlap family grouping rules (edit these to regroup models).
OVERLAP_FAMILY_RULES = {
    ("gpt2", "babylm-baseline"): "GPT-2",
    "babyberta": "BabyBERTa",
    "elc_bert": "ELC-BERT",
    "elc-bert": "ELC-BERT",
    "ltg-bert": "LTG-BERT",
    "ltgbert": "LTG-BERT",
    "gpt-bert-babylm": "GPT-BERT (MLM)",
    "babylm-baseline-10m-gpt-bert": {
        "ar": "GPT-BERT (AR)",
        "mlm": "GPT-BERT (MLM)",
    },
    "babylm-baseline-100m-gpt-bert": {
        "ar": "GPT-BERT (AR)",
        "mlm": "GPT-BERT (MLM)",
    },
    "roberta-base-strict": "RoBERTa",
    "roberta-med-small": "RoBERTa (nyu-mll variants)",
    "roberta-base-": "RoBERTa (nyu-mll variants)",
    "roberta-base": "RoBERTa",
    "roberta-large": "RoBERTa",
    "babyllama": "BabyLLaMA",
    "baby-llama": "BabyLLaMA",
    "opt-125m-strict": "OPT-125M",
    "opt-125m": "OPT-125M",
    "opt": "OPT-125M",
    "gpt2": "GPT-2",
    "t5-base-strict": "t5-base",
    "t5": "t5-base",
}
OVERLAP_FAMILY_FALLBACK = {
    "mlm": "Other MLM",
    "ar": "Other AR",
    "s2s": "Other S2S",
    "seq2seq": "Other S2S",
}

# ANALYSIS SUBSET — change TARGET_SPEAKER to compare child vs mother.
TARGET_SPEAKER = "child"  # "child" | "mother"

# Canonical paper subset writes to paper_results/; other speaker gets its own directory.
OUT_DIR = (
    Path("./figures")
    if TARGET_SPEAKER == "child"
    else Path(f"./figures_{TARGET_SPEAKER}")
)

# Visualization color/marker constants.
arch_colors  = {"MLM": "#1f77b4", "AR": "#ff7f0e", "S2S": "#2ca02c"}
arch_markers = {"MLM": "o", "AR": "s", "S2S": "^"}
child_color     = "#89cff0"  # baby blue for child human reference
caretaker_color = "#ffc0cb"  # pink for caretaker human reference

OUT_DIR.mkdir(parents=True, exist_ok=True)

## Load Results

In [3]:
# Load model configurations
with open('model_configs.json', 'r') as f:
    model_configs = json.load(f)

import glob as _glob

def _load_analytical(base_dir):
    """Load analytical_overlap_summary.csv files for all models under base_dir.

    Returns (acc_df, overlap_df) where both share the analytical schema:
      model_name, model_type, child_name, speaker, N, S, emp_bias,
      empirical, naive_predicted, accuracy
    """
    frames = []
    for csv_path in sorted(_glob.glob(f'{base_dir}/*/*/analytical_overlap_summary.csv')):
        frames.append(pd.read_csv(csv_path))
    if not frames:
        return None, None
    df = pd.concat(frames, ignore_index=True)
    # Normalize to short model names (strip HuggingFace org prefix if present).
    df['model_name'] = df['model_name'].map(lambda n: str(n).split('/')[-1].strip())
    return df, df


# All analysis uses Experiment 2 (discourse) results from results/overlap.
discourse_accuracy, discourse_overlap = _load_analytical('./results/overlap')

# Load human baseline
human_baseline = pd.read_csv('./results/overlap/overlap_human_baseline.csv')

print(f"Discourse overlap rows: {len(discourse_overlap) if discourse_overlap is not None else 0}")
print(f"Human baseline rows: {len(human_baseline)}")

Discourse overlap rows: 1176
Human baseline rows: 24


## Helper Functions

In [4]:
def _sig_stars_from_p(p):
    if pd.isna(p):
        return "na"
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"

def _model_display_key(df):
    """Type-qualified model display key used to prevent cross-type pooling."""
    if df is None:
        return None
    if "model_type" in df.columns:
        return df["model_type"].astype(str) + "::" + df["model_name"].astype(str)
    return df["model_name"].astype(str)

def _to_overlap_long(overlap_df):
    """Convert analytical overlap DataFrame to long form for plotting and stats.

    Analytical data has one row per (model, dyad, speaker) — no method column.
    """
    base_cols = ["model_name", "model_type", "speaker", "naive", "adj", "empirical", "child_name", "emp_bias"]
    if overlap_df is None or overlap_df.empty:
        return pd.DataFrame(columns=base_cols)

    naive_col = "naive_predicted" if "naive_predicted" in overlap_df.columns else "naive_zipf_predicted"
    adj_col = "adj_predicted" if "adj_predicted" in overlap_df.columns else "adj_zipf_predicted"

    if {naive_col, "empirical"}.issubset(overlap_df.columns):
        keep_cols = ["model_name", "speaker", naive_col, "empirical"]
        if "model_type" in overlap_df.columns:
            keep_cols.insert(1, "model_type")
        if "child_name" in overlap_df.columns:
            keep_cols.append("child_name")
        if "emp_bias" in overlap_df.columns:
            keep_cols.append("emp_bias")
        if adj_col in overlap_df.columns:
            keep_cols.append(adj_col)

        out = overlap_df[keep_cols].copy().rename(columns={naive_col: "naive", adj_col: "adj"})
        if "model_type" not in out.columns:
            out["model_type"] = "unknown"
        if "child_name" not in out.columns:
            out["child_name"] = "all"
        if "emp_bias" not in out.columns:
            out["emp_bias"] = np.nan
        if "adj" not in out.columns:
            out["adj"] = np.nan
        return out[["model_name", "model_type", "speaker", "naive", "adj", "empirical", "child_name", "emp_bias"]].dropna(subset=["empirical"])

    return pd.DataFrame(columns=base_cols)

def _compute_overlap_sig_table(overlap_df):
    """Paired t-test (empirical vs naive) per (model, speaker). No method split."""
    long_df = _to_overlap_long(overlap_df)
    rows = []
    key_df = long_df[["model_name", "model_type"]].drop_duplicates() if not long_df.empty else pd.DataFrame(columns=["model_name", "model_type"])
    for _, mrow in key_df.iterrows():
        model_name = mrow["model_name"]
        model_type = mrow["model_type"]
        for speaker in SPEAKERS:
            sub = long_df[
                (long_df["model_name"] == model_name)
                & (long_df["model_type"] == model_type)
                & (long_df["speaker"] == speaker)
            ]
            p = np.nan
            n_samples = len(sub.dropna(subset=["empirical", "naive"]))
            if n_samples > 1:
                try:
                    _, p = ttest_rel(sub["empirical"], sub["naive"], nan_policy="omit")
                except Exception:
                    p = np.nan
            rows.append({
                "model_name": model_name,
                "model_type": model_type,
                "speaker": speaker,
                "n_pairs": n_samples,
                "p_value": p,
                "sig": _sig_stars_from_p(p),
            })
    return pd.DataFrame(rows)

def _plot_overlap_scatter(overlap_df, title):
    """Diagnostic scatter: empirical vs naive overlap, one panel per model."""
    long_df = _to_overlap_long(overlap_df)
    if long_df.empty or "speaker" not in long_df.columns:
        return

    long_df = long_df.copy()
    long_df["model_key"] = _model_display_key(long_df)
    long_df["speaker_label"] = long_df["speaker"].astype(str).str.strip().str.lower().map({
        "child": "Child", "mother": "Mother",
    }).fillna(long_df["speaker"].astype(str))

    speaker_colors = {"Child": "#1f77b4", "Mother": "#ff7f0e"}
    speaker_markers = {"Child": "o", "Mother": "s"}

    model_list = sorted(long_df["model_key"].dropna().unique())
    if not model_list:
        return

    ncols = 3
    nrows = int(np.ceil(len(model_list) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows), squeeze=False)
    axes_flat = axes.flatten()

    min_xy = float(np.nanmin(np.r_[long_df["naive"].values, long_df["empirical"].values]))
    max_xy = float(np.nanmax(np.r_[long_df["naive"].values, long_df["empirical"].values]))

    for i, model_key in enumerate(model_list):
        ax = axes_flat[i]
        sub = long_df[long_df["model_key"] == model_key]
        for spk in ["Child", "Mother"]:
            group = sub[sub["speaker_label"] == spk]
            if group.empty:
                continue
            ax.scatter(group["naive"], group["empirical"],
                       marker=speaker_markers.get(spk, "o"), s=55, alpha=0.75,
                       color=speaker_colors.get(spk, "gray"),
                       edgecolors="black" if spk == "Mother" else "none", linewidths=0.5)

        p_text_parts = []
        for spk in ["Child", "Mother"]:
            group = sub[sub["speaker_label"] == spk]
            if len(group) <= 1:
                continue
            try:
                _, p_val = ttest_rel(group["empirical"], group["naive"], nan_policy="omit")
                p_str = "p<0.001" if p_val < 0.001 else f"p={p_val:.3f}" if p_val < 0.05 else f"p={p_val:.2f}"
                p_text_parts.append(f"{spk[0]}: {p_str}")
            except Exception:
                pass
        if p_text_parts:
            ax.text(0.02, 0.98, "\n".join(p_text_parts), transform=ax.transAxes, fontsize=7,
                    verticalalignment="top", bbox=dict(boxstyle="round", facecolor="white", alpha=0.8, edgecolor="gray", linewidth=0.5))

        if "emp_bias" in sub.columns:
            bias_parts = [f"{spk[0]}: {sub[sub['speaker_label']==spk]['emp_bias'].mean():.3f}"
                          for spk in ["Child", "Mother"] if not sub[sub['speaker_label']==spk].empty
                          and sub[sub['speaker_label']==spk]['emp_bias'].notna().any()]
            if bias_parts:
                ax.text(0.98, 0.02, "\n".join(bias_parts), transform=ax.transAxes, fontsize=7,
                        verticalalignment="bottom", horizontalalignment="right",
                        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8, edgecolor="gray", linewidth=0.5))

        ax.plot([min_xy, max_xy], [min_xy, max_xy], linestyle="--", linewidth=1, color="black")
        ax.set_xlim(min_xy, max_xy); ax.set_ylim(min_xy, max_xy)
        ax.set_title(model_key); ax.set_xlabel("Naive Overlap"); ax.set_ylabel("Empirical Overlap")
        from matplotlib.lines import Line2D
        ax.legend(handles=[
            Line2D([0], [0], marker="o", color="w", markerfacecolor=speaker_colors["Child"], markersize=7, label="Child"),
            Line2D([0], [0], marker="s", color="w", markerfacecolor=speaker_colors["Mother"], markeredgecolor="black", markeredgewidth=0.5, markersize=8, label="Mother"),
        ], loc="lower right", fontsize=6, frameon=True)

    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].axis("off")
    fig.suptitle(title, y=1.02, fontsize=14)
    plt.tight_layout()
    plt.show()

def _plot_accuracy_bars(accuracy_df, title, value_label="Accuracy",
                        sort_alphabetical=False, y_limits=(0, 1),
                        baseline_by_speaker=None, drop_human=False):
    if accuracy_df is None or accuracy_df.empty:
        return

    acc = accuracy_df.copy()
    if "model_name" not in acc.columns or "speaker" not in acc.columns:
        return

    speaker_map = {"child": "child", "mother": "mother", "mom": "mother", "mum": "mother"}
    acc["speaker"] = acc["speaker"].astype(str).str.strip().str.lower().map(speaker_map).fillna(acc["speaker"].astype(str).str.strip().str.lower())

    acc_col = next((c for c in ["accuracy", "accuracy_argmax", "accuracy_deterministic"] if c in acc.columns), None)
    if acc_col is None:
        return

    if drop_human:
        acc = acc[acc["model_name"].astype(str).str.lower() != "human"]

    plot_df = acc.groupby(["model_name", "speaker"], as_index=False)[acc_col].mean(numeric_only=True).rename(columns={acc_col: "value"})

    model_order = (sorted(plot_df["model_name"].dropna().unique()) if sort_alphabetical
                   else plot_df.groupby("model_name")["value"].mean().sort_values(ascending=False).index.tolist())

    fig_height = max(5, 0.25 * len(model_order) + 2)
    fig, axes = plt.subplots(1, 2, figsize=(14, fig_height), sharey=True)

    for idx, speaker in enumerate(SPEAKERS):
        ax = axes[idx]
        sub = plot_df[plot_df["speaker"] == speaker].copy()
        if sub.empty:
            ax.axis("off")
            continue
        sns.barplot(data=sub, y="model_name", x="value", order=model_order, ax=ax, errorbar=None, color="#1f77b4")
        if y_limits is not None:
            ax.set_xlim(y_limits[0], y_limits[1])
        if baseline_by_speaker is not None and speaker in baseline_by_speaker and pd.notna(baseline_by_speaker[speaker]):
            ax.axvline(x=float(baseline_by_speaker[speaker]), color="gray", linestyle="--", alpha=0.9, linewidth=1.5)
        ax.set_title(f"{speaker.capitalize()} {value_label}")
        ax.set_ylabel("Model"); ax.set_xlabel(value_label)

    fig.suptitle(title, y=1.02, fontsize=14)
    plt.tight_layout()
    plt.show()

def _plot_experiment_bars(df, title, y_col="accuracy", x_label=None):
    if df is None or df.empty:
        return

    work_df = df.copy()
    if "model_name" not in work_df.columns or "speaker" not in work_df.columns:
        return

    if "model_type" not in work_df.columns:
        work_df["model_type"] = "unknown"
    work_df["model_key"] = work_df["model_type"].astype(str) + "::" + work_df["model_name"].astype(str)

    speaker_map = {"child": "child", "mother": "mother", "mom": "mother", "mum": "mother"}
    work_df["speaker"] = work_df["speaker"].astype(str).str.strip().str.lower().map(speaker_map).fillna(work_df["speaker"].astype(str).str.strip().str.lower())

    val_col = (next((c for c in ["accuracy", "accuracy_argmax", "accuracy_deterministic"] if c in work_df.columns), None)
               if y_col == "accuracy" else (y_col if y_col in work_df.columns else None))
    if val_col is None:
        return

    long_df = work_df.groupby(["model_key", "speaker"], as_index=False)[val_col].mean(numeric_only=True).rename(columns={val_col: "value"})

    if x_label is None:
        x_label = "Accuracy" if y_col == "accuracy" else "TPR"

    human_baseline_vals = {}
    mk = long_df["model_key"].astype(str).str.lower()
    is_human = mk.str.endswith("::human") | (mk == "human")
    if is_human.any():
        for spk in SPEAKERS:
            sub = long_df[(is_human) & (long_df["speaker"] == spk)]
            if not sub.empty:
                human_baseline_vals[spk] = sub["value"].mean()
        long_df = long_df[~is_human]

    model_order = long_df.groupby("model_key")["value"].mean().sort_values(ascending=False).index.tolist()
    fig_height = max(5, 0.25 * len(model_order) + 2)
    fig, axes = plt.subplots(1, 2, figsize=(14, fig_height), sharey=True)
    baseline_by_speaker = {"child": 0.535, "mother": 0.546}

    for idx, speaker in enumerate(SPEAKERS):
        ax = axes[idx]
        sub = long_df[long_df["speaker"] == speaker].copy()
        if sub.empty:
            ax.axis("off")
            continue
        sns.barplot(data=sub, y="model_key", x="value", order=model_order, ax=ax, errorbar=None, color="#1f77b4")
        if y_col == "accuracy":
            ax.axvline(x=baseline_by_speaker.get(speaker, 0.5), color="gray", linestyle="--", alpha=0.9, linewidth=1.5, label="Majority Determiner")
            ax.set_xlim(0, 1)
        elif speaker in human_baseline_vals:
            ax.axvline(x=human_baseline_vals[speaker], color="gray", linestyle="--", alpha=0.9, linewidth=1.5, label="Human Avg")
        ax.set_title(f"{speaker.capitalize()} {x_label}")
        ax.set_ylabel("Model"); ax.set_xlabel(x_label)
        handles, labels = ax.get_legend_handles_labels()
        if handles:
            by_label = dict(zip(labels, handles))
            ax.legend(by_label.values(), by_label.keys(), fontsize=9, loc="lower right")

    fig.suptitle(title, y=1.02, fontsize=14)
    plt.tight_layout()
    plt.show()

def _print_experiment_tables(overlap_df, acc_df, title="Experiment"):
    import scipy.stats as stats

    def _norm_model(name):
        return str(name).strip().split('/')[-1].lower()

    if overlap_df is None or overlap_df.empty:
        return

    long_overlap = _to_overlap_long(overlap_df)
    overlap = long_overlap.copy()
    if 'model_type' not in overlap.columns:
        overlap['model_type'] = 'unknown'
    overlap['model_name_norm'] = overlap['model_name'].map(_norm_model)

    acc = None
    if acc_df is not None and not acc_df.empty:
        acc = acc_df.copy()
        if 'model_type' not in acc.columns:
            acc['model_type'] = 'unknown'
        acc['model_name_norm'] = acc['model_name'].map(_norm_model)

    acc_col = next((c for c in ["accuracy", "accuracy_argmax", "accuracy_deterministic"] if acc is not None and c in acc.columns), None)

    for speaker in SPEAKERS:
        part_overlap = overlap[overlap['speaker'] == speaker]
        if part_overlap.empty:
            continue

        if acc is not None and acc_col is not None:
            part_acc = (
                acc[acc['speaker'] == speaker]
                .groupby(['model_type', 'model_name_norm'], as_index=False)[acc_col]
                .mean(numeric_only=True)
            )
        else:
            part_acc = pd.DataFrame(columns=['model_type', 'model_name_norm', 'accuracy'])
            acc_col = 'accuracy'

        rows = []
        for (model_type, model_name_norm), group in part_overlap.groupby(['model_type', 'model_name_norm']):
            emp = group['empirical'].dropna()
            naive = group['naive'].dropna()
            adj = group['adj'].dropna() if 'adj' in group.columns else pd.Series(dtype=float)
            common_idx = emp.index.intersection(naive.index)
            emp_valid = emp.loc[common_idx]
            naive_valid = naive.loc[common_idx]
            n_samples = len(common_idx)
            mean_emp = emp_valid.mean() if not emp_valid.empty else np.nan
            mean_naive = naive_valid.mean() if not naive_valid.empty else np.nan
            common_adj_idx = emp.index.intersection(adj.index)
            mean_adj = adj.loc[common_adj_idx].mean() if not adj.empty and len(common_adj_idx) > 0 else np.nan
            emp_bias_col = group['emp_bias'].dropna() if 'emp_bias' in group.columns else pd.Series(dtype=float)
            mean_emp_bias = emp_bias_col.mean() if not emp_bias_col.empty else np.nan
            p_val = np.nan
            if n_samples > 1:
                try:
                    _, p_val = stats.ttest_rel(emp_valid, naive_valid)
                except Exception:
                    pass
            sig = '***' if (pd.notna(p_val) and p_val < 0.001) else '**' if (pd.notna(p_val) and p_val < 0.01) else '*' if (pd.notna(p_val) and p_val < 0.05) else 'ns'
            acc_val = part_acc[(part_acc['model_type'] == model_type) & (part_acc['model_name_norm'] == model_name_norm)][acc_col]
            rows.append({
                'model_name': model_name_norm, 'model_type': model_type,
                'n_pairs': n_samples, 'emp_bias': mean_emp_bias,
                'empirical_overlap': mean_emp, 'naive_predicted_overlap': mean_naive,
                'adj_predicted_overlap': mean_adj,
                'determiner_accuracy': acc_val.iloc[0] if not acc_val.empty else np.nan,
                'sig': sig
            })

        out_df = pd.DataFrame(rows).sort_values(by='determiner_accuracy', ascending=False)
        print(f"\n{title} - Speaker: {speaker.capitalize()}")
        print("=" * 140)
        output = out_df.rename(columns={
            'model_name': 'Model Name', 'model_type': 'Model Type',
            'n_pairs': 'N Pairs', 'emp_bias': 'Emp. Bias',
            'empirical_overlap': 'Empirical', 'naive_predicted_overlap': 'Naive Predicted',
            'adj_predicted_overlap': 'Adj Predicted', 'determiner_accuracy': 'Det. Accuracy',
            'sig': 'Sig. Diff. (Naive vs Emp)'
        }).copy()
        output = output[['Model Name', 'Model Type', 'N Pairs', 'Emp. Bias', 'Empirical', 'Naive Predicted', 'Adj Predicted', 'Det. Accuracy', 'Sig. Diff. (Naive vs Emp)']]
        for col in ['Emp. Bias', 'Empirical', 'Naive Predicted', 'Adj Predicted', 'Det. Accuracy']:
            output[col] = output[col].apply(lambda x: f"{x:.3f}" if pd.notna(x) else "N/A")
        print(output.to_string(index=False))
        print("-" * 140)


# Paper-specific helper functions.

def _fmt_p(p):
    if pd.isna(p):
        return "NA"
    if p < 0.001:
        return "<0.001"
    return f"{p:.3f}"


def _p_pass(p, alpha=PASS_ALPHA):
    return bool(pd.notna(p) and p > alpha)


def _pass_mark(p, alpha=PASS_ALPHA):
    return "check" if _p_pass(p, alpha) else "cross"


def _fmt_mean_sd(mean_val, sd_val, nd=3):
    if pd.isna(mean_val):
        return "NA"
    if pd.isna(sd_val):
        return f"{mean_val:.{nd}f} (NA)"
    return f"{mean_val:.{nd}f} ({sd_val:.{nd}f})"


def _norm_model_name(name):
    return str(name).split("/")[-1].strip()


def _arch_abbrev(model_type):
    t = str(model_type).lower()
    if t == "mlm":
        return "MLM"
    if t in {"ar", "autoregressive"}:
        return "AR"
    if t in {"seq2seq", "s2s"}:
        return "S2S"
    return str(model_type).upper()


def _unique_plain_labels(labels, archs=None):
    labels_list = list(labels)
    arch_list = list(archs) if archs is not None else None
    totals = {}
    for value in labels_list:
        totals[value] = totals.get(value, 0) + 1
    out = []
    per_label_counts = {}
    for idx, value in enumerate(labels_list):
        if totals.get(value, 0) > 1:
            if arch_list is not None and idx < len(arch_list) and pd.notna(arch_list[idx]):
                out.append(f"{value} ({arch_list[idx]})")
            else:
                per_label_counts[value] = per_label_counts.get(value, 0) + 1
                out.append(f"{value} ({per_label_counts[value]})")
        else:
            out.append(value)
    return out


# Build summary tables used by paper artifacts.

def _build_overlap_summary(overlap_df):
    cols = ["model", "model_type", "arch", "mean_emp_overlap", "sd_emp_overlap",
            "mean_pred_overlap", "sd_pred_overlap", "emp_bias", "p_DxN"]
    if overlap_df is None or overlap_df.empty:
        return pd.DataFrame(columns=cols)

    long_df = _to_overlap_long(overlap_df)
    if long_df.empty:
        return pd.DataFrame(columns=cols)

    work = long_df.copy()
    if "speaker" in work.columns and (work["speaker"] == TARGET_SPEAKER).any():
        work = work[work["speaker"] == TARGET_SPEAKER]

    work["model_type"] = work.get("model_type", "unknown")
    work["model_name_norm"] = work["model_name"].map(_norm_model_name)

    rows = []
    for (model_name_norm, model_type), group in work.groupby(["model_name_norm", "model_type"]):
        sample = group[["empirical", "naive"]].dropna()
        p_dx = np.nan
        if len(sample) > 1:
            try:
                _, p_dx = ttest_rel(sample["empirical"], sample["naive"])
            except Exception:
                p_dx = np.nan
        rows.append({
            "model": model_name_norm,
            "model_type": model_type,
            "arch": _arch_abbrev(model_type),
            "mean_emp_overlap": group["empirical"].mean(),
            "sd_emp_overlap": group["empirical"].std(ddof=1),
            "mean_pred_overlap": group["naive"].mean(),
            "sd_pred_overlap": group["naive"].std(ddof=1),
            "emp_bias": group["emp_bias"].mean() if "emp_bias" in group.columns else np.nan,
            "p_DxN": p_dx,
        })
    return pd.DataFrame(rows, columns=cols)


def _build_acc_summary(acc_df):
    cols = ["model", "model_type", "det_accuracy"]
    if acc_df is None or acc_df.empty:
        return pd.DataFrame(columns=cols)

    work = acc_df.copy()
    if "speaker" in work.columns and (work["speaker"] == TARGET_SPEAKER).any():
        work = work[work["speaker"] == TARGET_SPEAKER]

    det_col = next((c for c in ["accuracy", "accuracy_argmax", "accuracy_deterministic", "accuracy_sample"]
                    if c in work.columns), None)
    if det_col is None:
        return pd.DataFrame(columns=cols)

    work["model_type"] = work.get("model_type", "unknown")
    work["model_name_norm"] = work["model_name"].map(_norm_model_name)

    grouped = work.groupby(["model_name_norm", "model_type"], as_index=False)[det_col].mean(numeric_only=True)
    grouped = grouped.rename(columns={"model_name_norm": "model", det_col: "det_accuracy"})
    return grouped[cols]


def _baseline_overlap_stats(baseline_df, speaker_key):
    speaker_norm = baseline_df["speaker"].astype(str).str.lower().str.strip()
    if speaker_key == "child":
        subset = baseline_df[~speaker_norm.str.contains("_mot", na=False)].copy()
    else:
        subset = baseline_df[speaker_norm.str.contains("_mot", na=False)].copy()
    paired = subset[["empirical", "naive_zipf_predicted"]].dropna()
    p_dx = np.nan
    if len(paired) > 1:
        try:
            _, p_dx = ttest_rel(paired["empirical"], paired["naive_zipf_predicted"])
        except Exception:
            p_dx = np.nan
    return {
        "mean_emp": paired["empirical"].mean() if len(paired) else np.nan,
        "sd_emp": paired["empirical"].std(ddof=1) if len(paired) > 1 else np.nan,
        "mean_pred": paired["naive_zipf_predicted"].mean() if len(paired) else np.nan,
        "sd_pred": paired["naive_zipf_predicted"].std(ddof=1) if len(paired) > 1 else np.nan,
        "p_DxN": p_dx,
    }

def _format_test_cell(p):
    if pd.isna(p):
        return "--"
    mark = r"$\\checkmark$" if p > PASS_ALPHA else r"$\\times$"
    p_text = "<0.001" if p < 0.001 else f"{p:.3f}"
    return f"{mark} ({p_text})"


def _assign_family(row):
    """Map a model row to its family label using OVERLAP_FAMILY_RULES."""
    for key, val in OVERLAP_FAMILY_RULES.items():
        matches = (
            all(k in str(row["model_name"]).lower() for k in key)
            if isinstance(key, tuple)
            else key in str(row["model_name"]).lower()
        )
        if matches:
            if isinstance(val, dict):
                return val.get(str(row.get("model_type", "")).lower(), val.get("default", "Other"))
            return val
    return OVERLAP_FAMILY_FALLBACK.get(str(row.get("model_type", "")).lower(), "Other")


def _plot_family_overlap(overlap_df, title, out_path=None):
    """Family-grouped overlap scatter (human facets first, then model families)."""
    long = _to_overlap_long(overlap_df)
    if long.empty:
        print(f"No data for: {title}")
        return

    child_data = long[long["speaker"] == TARGET_SPEAKER].copy()
    if child_data.empty:
        print(f"No {TARGET_SPEAKER} data for: {title}")
        return
    child_data["model_name_norm"] = child_data["model_name"].map(_norm_model_name)
    child_data["family"] = child_data.apply(_assign_family, axis=1)

    # Build human reference facets from global human_baseline.
    human_df = pd.DataFrame(columns=["naive", "empirical", "facet_label"])
    if "human_baseline" in globals() and isinstance(human_baseline, pd.DataFrame):
        hb = human_baseline.copy()
        speaker_norm = hb["speaker"].astype(str).str.lower().str.strip()
        naive_col = "naive_zipf_predicted" if "naive_zipf_predicted" in hb.columns else "naive_predicted"
        if naive_col in hb.columns and "empirical" in hb.columns:
            hc = hb.loc[~speaker_norm.str.contains("_mot", na=False), [naive_col, "empirical"]].rename(columns={naive_col: "naive"})
            hc["facet_label"] = "Human | Child"
            hm = hb.loc[speaker_norm.str.contains("_mot", na=False), [naive_col, "empirical"]].rename(columns={naive_col: "naive"})
            hm["facet_label"] = "Human | Caretaker"
            human_df = pd.concat([hc, hm], ignore_index=True)

    human_color_map = {"Human | Child": child_color, "Human | Caretaker": caretaker_color}
    human_facets = [f for f in ["Human | Child", "Human | Caretaker"]
                    if not human_df.empty and f in human_df["facet_label"].values]

    family_order = [
        "BabyBERTa", "ELC_BERT", "ltg-bert",
        "GPT-BERT (MLM)", "GPT-BERT (AR)",
        "roberta", "roberta-base (nyu-mll variants)",
        "BabyLLaMA", "OPT-125M", "GPT-2", "t5-base",
        "Other MLM", "Other AR", "Other S2S", "Other",
    ]
    present = child_data["family"].dropna().unique().tolist()
    ordered = [f for f in family_order if f in present]
    ordered += sorted(f for f in present if f not in ordered)
    facet_labels = human_facets + ordered

    ncols = 3 if len(facet_labels) >= 3 else max(1, len(facet_labels))
    nrows = int(np.ceil(len(facet_labels) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.4 * ncols, 4.2 * nrows), squeeze=False)
    axes_flat = axes.flatten()

    for idx, family in enumerate(facet_labels):
        ax = axes_flat[idx]
        if family in human_facets:
            sub = human_df[human_df["facet_label"] == family]
            ax.scatter(sub["naive"], sub["empirical"], s=55,
                       color=human_color_map.get(family, "#9aa0a6"), edgecolors="black", linewidths=0.5)
        else:
            group = child_data[child_data["family"] == family].copy()
            if group.empty:
                ax.axis("off")
                continue
            group["model_key"] = group["model_type"].astype(str) + "::" + group["model_name_norm"]
            model_keys = group["model_key"].unique().tolist()
            palette = sns.color_palette("tab10", n_colors=max(len(model_keys), 3))
            model_colors = {k: palette[i % len(palette)] for i, k in enumerate(model_keys)}
            for mkey in model_keys:
                sub = group[group["model_key"] == mkey]
                ax.scatter(sub["naive"], sub["empirical"], s=30,
                           color=model_colors[mkey], alpha=0.75, edgecolors="black", linewidths=0.2, zorder=2,
                           label=sub["model_name_norm"].iloc[0])
            ax.legend(loc="lower right", fontsize=6, frameon=True, title="Models")

        ax.plot([0, 0.5], [0, 0.5], linestyle="--", color="red", linewidth=1)
        ax.set_xlim(0, 0.5); ax.set_ylim(0, 0.5)
        ax.set_title(family); ax.set_xlabel("Predicted overlap"); ax.set_ylabel("Empirical overlap")

    for j in range(len(facet_labels), len(axes_flat)):
        axes_flat[j].axis("off")

    fig.suptitle(title, y=1.02, fontsize=12)
    fig.tight_layout()
    if out_path is not None:
        fig.savefig(out_path, dpi=300, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved: {out_path}")
    else:
        plt.show()

In [5]:
# Load TPR summaries produced by human_baseline.ipynb.
tpr_model_summary = pd.read_csv("./results/tpr/tpr_model_summary_analytical.csv")
print(f"Loaded TPR model summary: {len(tpr_model_summary)} models")
tpr_human_summary = pd.read_csv("./results/tpr/tpr_human_summary.csv")

ADULT_TPR_POPULATION_MEAN = tpr_human_summary["adult_tpr_population_mean"].dropna().iloc[0]

# Build tpr_summary_use for the figure (one row per model: model, arch, mean_tpr, sd_tpr, p_TPR).
tpr_summary_use = (
    tpr_model_summary
    .rename(columns={
        "model_short": "model",
        "mean_model_tpr": "mean_tpr",
        "sd_model_tpr": "sd_tpr",
        "p_1sample": "p_TPR",
    })
    [["model", "model_type", "mean_tpr", "sd_tpr", "p_TPR"]]
    .copy()
)
tpr_summary_use["arch"] = tpr_summary_use["model_type"].map(_arch_abbrev)

# Build tpr_sig for the cross-experiment summary (paired t-test vs Manchester children).
# model_name is already the short form matching overlap summary model_name.
tpr_sig = (
    tpr_model_summary
    .assign(model_name=tpr_model_summary["model_name"].map(_norm_model_name))
    .rename(columns={"p_1sample": "p_value"})  # same test as TPR_pass in figures
    [["model_name", "model_type", "p_value"]]
    .copy()
)
tpr_sig["speaker"] = "child"
tpr_sig["sig"] = tpr_sig["p_value"].apply(
    lambda p: "***" if pd.notna(p) and p < 0.001
    else "**" if pd.notna(p) and p < 0.01
    else "*" if pd.notna(p) and p < 0.05
    else "ns" if pd.notna(p)
    else "na"
)

# Human TPR stats (loaded from tpr_human_summary.csv, written by human_baseline.ipynb).
_child_row = tpr_human_summary[tpr_human_summary["speaker"] == "child"].iloc[0]
_mother_row = tpr_human_summary[tpr_human_summary["speaker"] == "mother"].iloc[0]
human_child_mean_tpr  = float(_child_row["mean_tpr"])
human_child_std_tpr   = float(_child_row["sd_tpr"])
human_mother_mean_tpr = float(_mother_row["mean_tpr"])
human_mother_std_tpr  = float(_mother_row["sd_tpr"])
child_tpr_pval  = float(_child_row["p_1sample"])
mother_tpr_pval = float(_mother_row["p_1sample"])

# Build overlap and accuracy summaries for the analysis section figures.
overlap_summary_use = _build_overlap_summary(discourse_overlap)
if not overlap_summary_use.empty:
    overlap_summary_use["DxN_pass"] = overlap_summary_use["p_DxN"].map(_p_pass)
    overlap_summary_use["model_arch_key"] = overlap_summary_use["model"] + "||" + overlap_summary_use["arch"]

acc_summary_use = _build_acc_summary(discourse_accuracy)

tpr_summary_use["TPR_pass"] = tpr_summary_use["p_TPR"].map(_p_pass)

# Build joint table and selected-model subset (used by full results table and paper figures).
joint = overlap_summary_use.merge(
    tpr_summary_use,
    on=["model", "model_type", "arch"],
    how="inner",
).merge(
    acc_summary_use[["model", "model_type", "det_accuracy"]],
    on=["model", "model_type"],
    how="left",
)
if not joint.empty:
    joint["DxN_pass"] = joint["p_DxN"].map(_p_pass)
    joint["TPR_pass"] = joint["p_TPR"].map(_p_pass)
    joint["model_arch_key"] = joint["model"] + "||" + joint["arch"]

selected_rows = []
if not joint.empty:
    for spec in MAIN_MODEL_SPECS:
        model_key = str(spec.get("model", "")).lower()
        arch_key = str(spec.get("arch", "")).upper()
        subset = joint[joint["model"].astype(str).str.lower() == model_key]
        if arch_key:
            subset = subset[subset["arch"].astype(str).str.upper() == arch_key]
        if not subset.empty:
            selected_rows.append(subset.iloc[0])
selected = pd.DataFrame(selected_rows)
selected_keys = selected["model_arch_key"].tolist() if not selected.empty else []

Loaded TPR model summary: 49 models


# Analysis Workflow

This section provides a clean, top-to-bottom analysis pipeline with four parts:
1. Experiment 1: Human data validation
2. Experiment 2: Discourse
3. Experiment 3: TPR
4. Final cross-experiment significance summary

## Experiment 1: Human data validation

This section reproduces Goldin-Meadow & Yang (2017) with our updated text processing pipeline.

In [6]:
# Human overlap summary: mean (SD) by speaker group.
hb = human_baseline.copy()
speaker_norm = hb["speaker"].astype(str).str.lower().str.strip()

groups = [
    ("Children", ~speaker_norm.str.contains("_mot", na=False)),
    ("Caretakers", speaker_norm.str.contains("_mot", na=False)),
]

rows = []
for label, mask in groups:
    sub = hb.loc[mask, ["empirical", "naive_zipf_predicted", "emp_bias"]].dropna()

    emp_mean = sub["empirical"].mean()
    emp_sd = sub["empirical"].std(ddof=1)
    pred_mean = sub["naive_zipf_predicted"].mean()
    pred_sd = sub["naive_zipf_predicted"].std(ddof=1)
    bias_mean = sub["emp_bias"].mean()
    bias_sd = sub["emp_bias"].std(ddof=1)

    rows.append({
        "Group": label,
        "n": len(sub),
        "Bias": f"{bias_mean:.3f} ({bias_sd:.3f})" if len(sub) > 1 else "NA",
        "Empirical": f"{emp_mean:.3f} ({emp_sd:.3f})" if len(sub) > 1 else "NA",
        "Predicted": f"{pred_mean:.3f} ({pred_sd:.3f})" if len(sub) > 1 else "NA",
    })

summary_df = pd.DataFrame(rows)

print("Human overlap summary (Manchester)")
print(summary_df.to_string(index=False))

Human overlap summary (Manchester)
     Group  n          Bias     Empirical     Predicted
  Children 12 0.834 (0.042) 0.260 (0.067) 0.249 (0.112)
Caretakers 12 0.815 (0.021) 0.309 (0.041) 0.329 (0.071)


In [7]:
# Human baseline validation tests (paired t-tests vs naive_zipf_predicted, Pearson r vs empirical, and bias tests).
hb = human_baseline.copy()
speaker_norm = hb["speaker"].astype(str).str.lower().str.strip()

child_mask = ~speaker_norm.str.contains("_mot", na=False)
caretaker_mask = speaker_norm.str.contains("_mot", na=False)

child_pairs = hb.loc[child_mask, ["empirical", "naive_zipf_predicted"]].dropna()
caretaker_pairs = hb.loc[caretaker_mask, ["empirical", "naive_zipf_predicted"]].dropna()

child_corr_df = hb.loc[child_mask, ["r", "empirical"]].dropna()
caretaker_corr_df = hb.loc[caretaker_mask, ["r", "empirical"]].dropna()

print("Paired t-test: empirical vs naive_zipf_predicted")
if len(child_pairs) > 1:
    t_child, p_child = ttest_rel(child_pairs["empirical"], child_pairs["naive_zipf_predicted"])
    print(f"  Children:   t={t_child:+.4f}, p={p_child:.6f}, n={len(child_pairs)}")
else:
    print("  Children:   insufficient data")

if len(caretaker_pairs) > 1:
    t_caretaker, p_caretaker = ttest_rel(caretaker_pairs["empirical"], caretaker_pairs["naive_zipf_predicted"])
    print(f"  Caretakers: t={t_caretaker:+.4f}, p={p_caretaker:.6f}, n={len(caretaker_pairs)}")
else:
    print("  Caretakers: insufficient data")

print("\nPearson correlation: r (token/type ratio) vs empirical")
if len(child_corr_df) > 1:
    r_child, p_r_child = pearsonr(child_corr_df["r"], child_corr_df["empirical"])
    print(f"  Children: mean S/N={child_corr_df['r'].mean():+.4f}, r={r_child:+.4f}, p={p_r_child:.6f}, n={len(child_corr_df)}")
else:
    print("  Children:   insufficient data")

if len(caretaker_corr_df) > 1:
    r_caretaker, p_r_caretaker = pearsonr(caretaker_corr_df["r"], caretaker_corr_df["empirical"])
    print(f"  Caretakers: mean S/N={caretaker_corr_df['r'].mean():+.4f}, r={r_caretaker:+.4f}, p={p_r_caretaker:.6f}, n={len(caretaker_corr_df)}")
else:
    print("  Caretakers: insufficient data")

print(f"\nBias tests against UNIV_BIAS = {UNIV_BIAS:.3f}")
bias_child_df = hb.loc[child_mask, ["speaker", "emp_bias"]].dropna().copy()
bias_caretaker_df = hb.loc[caretaker_mask, ["speaker", "emp_bias"]].dropna().copy()

if len(bias_child_df) > 1:
    t_bias_child, p_bias_child = ttest_1samp(bias_child_df["emp_bias"], UNIV_BIAS)
    print(f"  Children vs UNIV_BIAS:   t={t_bias_child:+.4f}, p={p_bias_child:.6f}, n={len(bias_child_df)}")
else:
    print("  Children vs UNIV_BIAS:   insufficient data")

if len(bias_caretaker_df) > 1:
    t_bias_caretaker, p_bias_caretaker = ttest_1samp(bias_caretaker_df["emp_bias"], UNIV_BIAS)
    print(f"  Caretakers vs UNIV_BIAS: t={t_bias_caretaker:+.4f}, p={p_bias_caretaker:.6f}, n={len(bias_caretaker_df)}")
else:
    print("  Caretakers vs UNIV_BIAS: insufficient data")

bias_pairs = (
    hb.assign(
        child_name=hb["speaker"].astype(str).str.replace("_mot", "", regex=False),
        spk_type=np.where(hb["speaker"].astype(str).str.contains("_mot", na=False), "caretaker", "child"),
    )[ ["child_name", "spk_type", "emp_bias"] ]
    .dropna()
    .pivot_table(index="child_name", columns="spk_type", values="emp_bias", aggfunc="first")
    .dropna()
 )

if len(bias_pairs) > 1 and {"child", "caretaker"}.issubset(bias_pairs.columns):
    t_bias_paired, p_bias_paired = ttest_rel(bias_pairs["child"], bias_pairs["caretaker"])
    print(f"  Children vs Caretakers:  t={t_bias_paired:+.4f}, p={p_bias_paired:.6f}, n={len(bias_pairs)}")
else:
    print("  Children vs Caretakers:  insufficient paired data")

print(f"\nPaired t-test: children's vs caretakers' empirical overlap")
empirical_pairs = (
    hb.assign(
        child_name=hb["speaker"].astype(str).str.replace("_mot", "", regex=False),
        spk_type=np.where(hb["speaker"].astype(str).str.contains("_mot", na=False), "caretaker", "child"),
    )[ ["child_name", "spk_type", "empirical"] ]
    .dropna()
    .pivot_table(index="child_name", columns="spk_type", values="empirical", aggfunc="first")
    .dropna()
 )

if len(empirical_pairs) > 1 and {"child", "caretaker"}.issubset(empirical_pairs.columns):
    t_emp_paired, p_emp_paired = ttest_rel(empirical_pairs["child"], empirical_pairs["caretaker"])
    print(f"  Children vs Caretakers:  t={t_emp_paired:+.4f}, p={p_emp_paired:.6f}, n={len(empirical_pairs)}")
else:
    print("  Children vs Caretakers:  insufficient paired data")

Paired t-test: empirical vs naive_zipf_predicted
  Children:   t=+0.6334, p=0.539416, n=12
  Caretakers: t=-1.2940, p=0.222187, n=12

Pearson correlation: r (token/type ratio) vs empirical
  Children: mean S/N=+4.4021, r=+0.8909, p=0.000101, n=12
  Caretakers: mean S/N=+6.4216, r=+0.7200, p=0.008278, n=12

Bias tests against UNIV_BIAS = 0.820
  Children vs UNIV_BIAS:   t=+1.1337, p=0.281026, n=12
  Caretakers vs UNIV_BIAS: t=-0.8154, p=0.432120, n=12


  Children vs Caretakers:  t=+2.0628, p=0.063555, n=12

Paired t-test: children's vs caretakers' empirical overlap
  Children vs Caretakers:  t=-3.6251, p=0.003990, n=12


In [8]:
# Validation of Yang & Valian (2020)'s original results.
children_r =  [4.06, 5.22, 4.36, 11.60, 3.09, 3.01, 3.51, 5.78, 4.78, 3.30, 4.13, 6.81]
children_overlap = [0.33, 0.32, 0.33, 0.46, 0.22, 0.21, 0.24, 0.33, 0.29, 0.26, 0.26, 0.37]

mother_r = [9.18, 8.62, 6.42, 8.57, 8.55, 4.77, 5.04, 5.67, 5.74, 5.77, 7.71, 7.91]
mother_overlap = [0.40, 0.34, 0.35, 0.40, 0.30, 0.30, 0.27, 0.36, 0.31, 0.31, 0.35, 0.35]

r_child, p_child = pearsonr(children_r, children_overlap)
r_mother, p_mother = pearsonr(mother_r, mother_overlap)

print(f'Yang & Valian (2020) — Pearson r: tokens/types vs observed overlap (n=12)')
print(f'  Children:   r={r_child:+.4f}, p={p_child:.6f}')
print(f'  Caretakers: r={r_mother:+.4f}, p={p_mother:.6f}')

Yang & Valian (2020) — Pearson r: tokens/types vs observed overlap (n=12)
  Children:   r=+0.9133, p=0.000033
  Caretakers: r=+0.6412, p=0.024626


In [9]:
# Human baseline validation table (Tab. manchester-validation). TPR/n_TPR use the UNRESTRICTED noun-set baseline (matches reported results and model case counts).
_tpr_raw = pd.read_csv("results/tpr/tpr_human_baseline_unrestricted.csv")
_tpr_raw = _tpr_raw[_tpr_raw["speaker"].isin(["child", "mother"])]

_tpr_agg = _tpr_raw.groupby(["child_name", "speaker"]).agg(
    n_tpr=("n_total", "sum"),
    trans_to_the=("n_transitions_to_the", "sum"),
    trans_to_a=("n_transitions_to_a", "sum"),
).reset_index()
_n_total = _tpr_raw.groupby(["child_name", "speaker"])["n_total"].sum().reset_index(name="n_total_sum")
_tpr_agg = _tpr_agg.merge(_n_total, on=["child_name", "speaker"])
_tpr_agg["tpr"] = (_tpr_agg["trans_to_the"] + _tpr_agg["trans_to_a"]) / _tpr_agg["n_total_sum"]

_hb = human_baseline.copy()
_hb["child_name"] = _hb["speaker"].apply(lambda s: s.replace("_mot", ""))
_hb["spk_type"]   = _hb["speaker"].apply(lambda s: "mother" if "_mot" in s else "child")
_hb = _hb.merge(
    _tpr_agg[["child_name", "speaker", "n_tpr", "tpr"]],
    left_on=["child_name", "spk_type"], right_on=["child_name", "speaker"],
    how="left",
)

# Group-level DxN test p-values (paired t-test: empirical vs predicted overlap),
# computed from the same human_baseline used for the table so the caption never drifts.
_spk_norm    = human_baseline["speaker"].astype(str).str.lower().str.strip()
_child_dxn   = human_baseline.loc[~_spk_norm.str.contains("_mot", na=False), ["empirical", "naive_zipf_predicted"]].dropna()
_care_dxn    = human_baseline.loc[_spk_norm.str.contains("_mot", na=False),  ["empirical", "naive_zipf_predicted"]].dropna()
_p_dxn_child = ttest_rel(_child_dxn["empirical"], _child_dxn["naive_zipf_predicted"])[1]
_p_dxn_care  = ttest_rel(_care_dxn["empirical"],  _care_dxn["naive_zipf_predicted"])[1]

_dyad_order = ["Gail", "Dominic", "Becky", "Liz", "Carl", "Joel",
               "Ruth", "Aran", "Anne", "John", "Nicole", "Warren"]

_caption = (
    r"Types (\(N\)), tokens (\(S\)), determiner bias score, token/type ratio (\(r\)), and "
    r"observed (empirical) versus predicted overlap values for 12 children and their "
    r"corresponding caretakers in the Manchester corpus. Singular noun phrases were extracted "
    r"automatically using spaCy's noun chunker. At the group level, children's empirical "
    rf"overlap does not differ significantly from predicted overlap (\(p={_p_dxn_child:.3f}\)), and the same "
    rf"is true for caretakers (\(p={_p_dxn_care:.3f}\)), reproducing the formal productivity pattern reported "
    r"in prior work and validating the preprocessing pipeline used throughout the paper."
)

_lines = [
    r"\begin{table*}[]",
    r"\centering",
    r"\small",
    rf"\caption{{{_caption}}}",
    r"\label{tab:manchester-validation}",
    r"\begin{tabular}{llrrrrrrr}",
    r"\toprule",
    r"Dyad & Speaker & $|N|$ & $|S|$ & Bias & Empirical & Predicted & $n_\text{TPR}$ & TPR \\",
    r"\midrule",
]

for _dyad in _dyad_order:
    for _i, _spk in enumerate(["child", "mother"]):
        _mask = (_hb["child_name"] == _dyad) & (_hb["spk_type"] == _spk)
        _sub  = _hb[_mask]
        if _sub.empty:
            continue
        _r        = _sub.iloc[0]
        _spk_lbl  = "Child" if _spk == "child" else "Caretaker"
        _dyad_cell = rf"\multirow{{2}}{{*}}{{{_dyad}}}" if _i == 0 else "                         "
        _nt  = str(int(_r["n_tpr"])) if pd.notna(_r.get("n_tpr")) else "--"
        _tpr = f"{_r['tpr']:.3f}"   if pd.notna(_r.get("tpr"))   else "--"
        _lines.append(
            f"{_dyad_cell} & {_spk_lbl} & {int(_r['N'])} & {int(_r['S'])}"
            f" & {_r['emp_bias']:.3f} & {_r['empirical']:.3f}"
            f" & {_r['naive_zipf_predicted']:.3f} & {_nt} & {_tpr} \\\\"
        )

_lines += [r"\bottomrule", r"\end{tabular}", r"\end{table*}"]

_latex_str = "\n".join(_lines)
(OUT_DIR / "human_baseline_overlap.tex").write_text(_latex_str, encoding="utf-8")
print(f"Saved human_baseline_overlap.tex to {OUT_DIR}")
print(f"  DxN caption p-values: children p={_p_dxn_child:.3f}, caretakers p={_p_dxn_care:.3f}")

Saved human_baseline_overlap.tex to figures
  DxN caption p-values: children p=0.539, caretakers p=0.222


In [10]:
# Human-only overlap figure (two-facet standalone artifact).
human_df = pd.DataFrame(columns=["naive", "empirical", "facet_label"])
if "human_baseline" in globals() and isinstance(human_baseline, pd.DataFrame):
    hb = human_baseline.copy()
    speaker_norm = hb["speaker"].astype(str).str.lower().str.strip()
    naive_col = "naive_zipf_predicted" if "naive_zipf_predicted" in hb.columns else "naive_predicted"
    if naive_col in hb.columns and "empirical" in hb.columns:
        hc = hb.loc[~speaker_norm.str.contains("_mot", na=False), [naive_col, "empirical"]].rename(columns={naive_col: "naive"})
        hc["facet_label"] = "Human | Child"
        hm = hb.loc[speaker_norm.str.contains("_mot", na=False), [naive_col, "empirical"]].rename(columns={naive_col: "naive"})
        hm["facet_label"] = "Human | Caretaker"
        human_df = pd.concat([hc, hm], ignore_index=True)

human_color_map = {"Human | Child": child_color, "Human | Caretaker": caretaker_color}
if not human_df.empty:
    human_order = ["Human | Child", "Human | Caretaker"]
    plot_human = human_df[human_df["facet_label"].isin(human_order)].copy()
    fig, axes = plt.subplots(2, 1, figsize=(6.8, 9.0), squeeze=False)
    for idx, label in enumerate(human_order):
        ax = axes[idx, 0]
        sub = plot_human[plot_human["facet_label"] == label]
        if sub.empty:
            ax.axis("off")
            continue
        ax.scatter(sub["naive"], sub["empirical"], s=55,
                   color=human_color_map.get(label, "#9aa0a6"),
                   edgecolors="black", linewidths=0.5)
        ax.plot([0, 0.5], [0, 0.5], linestyle="--", color="red", linewidth=1)
        ax.set_xlim(0, 0.5)
        ax.set_ylim(0, 0.5)
        ax.set_title(label)
        ax.set_xlabel("Predicted overlap")
        ax.set_ylabel("Empirical overlap")
    fig.suptitle("Human overlap: child vs caretaker", y=1.02, fontsize=12)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "appendix_full_overlap_human_figure.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {OUT_DIR / 'appendix_full_overlap_human_figure.png'}")


Saved: figures/appendix_full_overlap_human_figure.png


## Experiment 2: Discourse

This section applies the same plotting/statistical template to discourse-conditioned results.

In [11]:
# Model family figure.
_plot_family_overlap(
    discourse_overlap,
    "DxN (CAC) experiment: Model overlap by family",
    OUT_DIR / "appendix_full_overlap_models_figure.png",
)


Saved: figures/appendix_full_overlap_models_figure.png


## Experiment 3: TPR

Full TPR analysis (human baseline validation, corpus equivalence tests,
and model comparisons) runs in `human_baseline.ipynb` and writes
`results/tpr/tpr_model_summary_analytical.csv` and `tpr_human_summary.csv`.
Those files were loaded in the setup cell above. The appendix figure is
generated in the paper artifacts section below.

In [12]:
# Appendix full TPR figure: SD bars and pass shading.
human_tpr_rows = []
if pd.notna(human_child_mean_tpr):
    human_tpr_rows.append({
        "model": "Child (Human)",
        "arch": "Human",
        "mean_tpr": human_child_mean_tpr,
        "sd_tpr": human_child_std_tpr,
        "TPR_pass": _p_pass(child_tpr_pval),
    })
if pd.notna(human_mother_mean_tpr):
    human_tpr_rows.append({
        "model": "Caretaker (Human)",
        "arch": "Human",
        "mean_tpr": human_mother_mean_tpr,
        "sd_tpr": human_mother_std_tpr,
        "TPR_pass": _p_pass(mother_tpr_pval),
    })

human_tpr_full = pd.DataFrame(human_tpr_rows)
model_tpr_full = tpr_summary_use.copy().reset_index(drop=True)

# Build display labels with architecture to avoid (2) suffixes for models only.
model_tpr_full["display_model"] = _unique_plain_labels(model_tpr_full["model"].tolist(), model_tpr_full["arch"].tolist())

# Ensure human rows appear together (children first, caretakers second).
if not human_tpr_full.empty:
    human_order = {"Child (Human)": 0, "Caretaker (Human)": 1}
    human_tpr_full["display_model"] = human_tpr_full["model"]
    human_tpr_full["_human_order"] = human_tpr_full["model"].map(human_order).fillna(99)
    human_tpr_full = human_tpr_full.sort_values("_human_order").drop(columns=["_human_order"])

# Sort models by TPR ascending, then append after the human rows.
model_tpr_full = model_tpr_full.sort_values("mean_tpr", ascending=True).reset_index(drop=True)
if not human_tpr_full.empty:
    full_tpr_plot = pd.concat([human_tpr_full, model_tpr_full], ignore_index=True)
else:
    full_tpr_plot = model_tpr_full.copy()

# Put the first rows at the top of the plot.
full_tpr_plot["ypos"] = np.arange(len(full_tpr_plot))[::-1]

fig, ax = plt.subplots(figsize=(8.2, max(6, 0.25 * len(full_tpr_plot) + 1.5)))
for _, row in full_tpr_plot.iterrows():
    if bool(row.get("TPR_pass", False)):
        ax.axhspan(row["ypos"] - 0.45, row["ypos"] + 0.45, color="#d9f2d9", alpha=0.6, zorder=0)
for _, row in full_tpr_plot.iterrows():
    if row.get("arch") == "Human":
        human_color = child_color if "Child" in str(row.get("model", "")) else caretaker_color
        ax.errorbar(
            x=row["mean_tpr"],
            y=row["ypos"],
            xerr=(row["sd_tpr"] if pd.notna(row["sd_tpr"]) else 0.0),
            fmt="D",
            color=human_color,
            ecolor=human_color,
            capsize=3,
            markersize=5,
            alpha=0.9,
            zorder=4,
        )
    else:
        ax.errorbar(
            x=row["mean_tpr"],
            y=row["ypos"],
            xerr=(row["sd_tpr"] if pd.notna(row["sd_tpr"]) else 0.0),
            fmt="o",
            color=arch_colors.get(row["arch"], "gray"),
            ecolor=arch_colors.get(row["arch"], "gray"),
            capsize=3,
            markersize=5,
            alpha=0.9,
            zorder=3,
        )
ax.axvline(ADULT_TPR_POPULATION_MEAN, linestyle="--", color="black", linewidth=1.5, label="Other-adult population mean")
ax.set_xlim(0, 1)
ax.set_yticks(full_tpr_plot["ypos"].tolist())
ax.set_yticklabels(full_tpr_plot["display_model"].tolist())
ax.set_xlabel("Mean model TPR (child, probabilistic)")
ax.set_ylabel("Model")
ax.set_title("Appendix: full TPR (with SD error bars)")
leg1 = ax.legend(handles=[Line2D([0], [0], marker="o", color="w", label=arch, markerfacecolor=color, markeredgecolor="black", markersize=8) for arch, color in arch_colors.items()] + [Line2D([0], [0], color="#d9f2d9", linewidth=8, label="TPR pass")], title="Architecture / Pass", loc="lower right")
ax.add_artist(leg1)
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig(OUT_DIR / "appendix_full_tpr_figure.png", dpi=300)
plt.close(fig)


## Final Summary: Cross-Experiment Significance

This section joins significance outcomes from isolated overlap, discourse overlap, and TPR into one comparison table.

In [13]:
# Cross-experiment significance summary: D×N overlap (Exp 2) and TPR (Exp 3).
# All results are derived from the discourse experiment (manchester_discourse_childes).

def _acc_per_speaker(acc_df, prefix):
    """Average accuracy per (model_name, model_type, speaker) from analytical data."""
    if acc_df is None or acc_df.empty:
        return pd.DataFrame(columns=["model_name", "model_type", "speaker", f"{prefix}_acc"])
    df = acc_df.copy()
    if "model_type" not in df.columns:
        df["model_type"] = "unknown"
    df["speaker"] = df["speaker"].astype(str).str.strip().str.lower()
    acc_col = next((c for c in ["accuracy", "accuracy_argmax", "accuracy_deterministic"] if c in df.columns), None)
    if acc_col is None:
        return pd.DataFrame(columns=["model_name", "model_type", "speaker", f"{prefix}_acc"])
    return df.groupby(["model_name", "model_type", "speaker"], as_index=False)[acc_col].mean().rename(columns={acc_col: f"{prefix}_acc"})

dis_sig = _compute_overlap_sig_table(discourse_overlap).rename(columns={"sig": "DxN_sig", "p_value": "DxN_p"})
dis_acc = _acc_per_speaker(discourse_accuracy, "disc")
dis_sig = dis_sig.merge(dis_acc, on=["model_name", "model_type", "speaker"], how="left")

tpr_sig_use = tpr_sig.rename(columns={"sig": "TPR_sig", "p_value": "TPR_p"})
summary_sig = dis_sig.merge(
    tpr_sig_use[["model_name", "model_type", "speaker", "TPR_sig"]],
    on=["model_name", "model_type", "speaker"],
    how="left",
)

print("" + "="*120)
print("Significance Summary — D×N Overlap and TPR (all results from Experiment 2: Discourse)")
print("="*120)

for spk in ["child", "mother"]:
    subset = summary_sig[summary_sig["speaker"].str.lower() == spk].copy()
    if subset.empty:
        continue

    spk_title = "CHILDREN" if spk == "child" else "MOTHERS"
    print(f"{spk_title}:")
    print("-" * 120)

    cols_to_print = ["model_name", "model_type", "disc_acc", "DxN_sig", "TPR_sig"]

    subset["disc_acc"] = subset["disc_acc"].apply(
        lambda x: f"{x:.3f}" if pd.notna(x) and not isinstance(x, str) else x
    )
    for c in cols_to_print:
        if c in subset.columns:
            subset[c] = subset[c].fillna("N/A")
        else:
            subset[c] = "N/A"

    if spk == "mother":
        subset["TPR_sig"] = "-"

    print(subset[cols_to_print].to_string(index=False))

print("" + "="*120)
print("Significance levels: *** p<0.001, ** p<0.01, * p<0.05, ns = not significant")
print("- = not applicable (mothers for TPR), N/A = data missing")
print("disc_acc: Discourse Determiner Accuracy")
print("DxN_sig:  D×N Overlap test — empirical vs Naive Zipf predicted (paired t-test)")
print("TPR_sig:  TPR test — model vs Manchester children (paired t-test, child speaker only)")
print("="*120)

Significance Summary — D×N Overlap and TPR (all results from Experiment 2: Discourse)
CHILDREN:
------------------------------------------------------------------------------------------------------------------------
                                model_name model_type disc_acc DxN_sig TPR_sig
    BabyLM-2026-Baseline-GPT2-Strict-Small         ar    0.700       *     ***
          BabyLM-2026-Baseline-GPT2-Strict         ar    0.653      **     ***
                            baby-llama-58m         ar    0.817       *      **
                       babyllama-100m-2024         ar    0.885      ns       *
                        babyllama-10m-2024         ar    0.870      ns       *
babylm-baseline-100m-gpt-bert-causal-focus         ar    0.898      ns       *
       babylm-baseline-100m-gpt-bert-mixed         ar    0.889      ns       *
                 babylm-baseline-100m-gpt2         ar    0.726       *     ***
 babylm-baseline-10m-gpt-bert-causal-focus         ar    0.861      ns  

In [14]:
# Generate appendix tables (all models with human baselines)

# Build full appendix overlap table with human baselines (DxN test includes pass + p-value).
full_table_outdir = OUT_DIR / "appendix_full_results_table"
appendix_rows = []

# Add human baselines (Children and Caretakers)
child_stats = _baseline_overlap_stats(human_baseline, "child")
caretaker_stats = _baseline_overlap_stats(human_baseline, "mother")

appendix_rows.append({
    "Model": "Children",
    "Arch.": "Human",
    "Bias": f"{0.834:.3f}",  # Hardcoded from analysis
    "Accuracy": "--",
    "Empirical (SD)": _fmt_mean_sd(child_stats["mean_emp"], child_stats["sd_emp"]),
    "Predicted (SD)": _fmt_mean_sd(child_stats["mean_pred"], child_stats["sd_pred"]),
    "TPR (SD)": _fmt_mean_sd(human_child_mean_tpr, human_child_std_tpr),
    "DxN test": _format_test_cell(child_stats["p_DxN"]),
    "TPR test": _format_test_cell(child_tpr_pval),
})

appendix_rows.append({
    "Model": "Caretakers",
    "Arch.": "Human",
    "Bias": f"{0.815:.3f}",  # Hardcoded from analysis
    "Accuracy": "--",
    "Empirical (SD)": _fmt_mean_sd(caretaker_stats["mean_emp"], caretaker_stats["sd_emp"]),
    "Predicted (SD)": _fmt_mean_sd(caretaker_stats["mean_pred"], caretaker_stats["sd_pred"]),
    "TPR (SD)": _fmt_mean_sd(human_mother_mean_tpr, human_mother_std_tpr),
    "DxN test": _format_test_cell(caretaker_stats["p_DxN"]),
    "TPR test": _format_test_cell(mother_tpr_pval),
})

# Add all models from overlap_summary_use
if not joint.empty:
    for _, row in joint.iterrows():
        empirical_str = _fmt_mean_sd(row.get("mean_emp_overlap"), row.get("sd_emp_overlap")) if pd.notna(row.get("mean_emp_overlap")) else "--"
        predicted_str = _fmt_mean_sd(row.get("mean_pred_overlap"), row.get("sd_pred_overlap"))
        bias_val = f"{row.get('emp_bias', np.nan):.3f}" if pd.notna(row.get("emp_bias")) else "--"
        acc_val = f"{row.get('det_accuracy', np.nan):.3f}" if pd.notna(row.get("det_accuracy")) else "--"
        tpr_str =  _fmt_mean_sd(row.get("mean_tpr"), row.get("sd_tpr")) if pd.notna(row.get("mean_tpr")) else "--"

        appendix_rows.append({
            "Model": row.get("model", "unknown"),
            "Arch.": row.get("arch", "unknown"),
            "Bias": bias_val,
            "Accuracy": acc_val,
            "Empirical (SD)": empirical_str,
            "Predicted (SD)": predicted_str,
            "TPR (SD)": tpr_str,
            "DxN test": _format_test_cell(row.get("p_DxN")),
            "TPR test": _format_test_cell(row.get("p_TPR")),
        })

# Create DataFrame with proper LaTeX column names
appendix_df = pd.DataFrame(appendix_rows)

main_columns = [
    "Model",
    "Arch.",
    "Bias",
    "Empirical (SD)",
    "Predicted (SD)",
    "TPR (SD)",
    "DxN test",
    "TPR test",
    "Accuracy",
]
human_rows = appendix_df[appendix_df["Arch."].astype(str).str.lower() == "human"]
model_rows = []
for spec in MAIN_MODEL_SPECS:
    model_key = str(spec.get("model", "")).lower()
    arch_key = str(spec.get("arch", "")).upper()
    subset = appendix_df[
        (appendix_df["Model"].astype(str).str.lower() == model_key)
        & (appendix_df["Arch."].astype(str).str.upper() == arch_key)
    ]
    if not subset.empty:
        model_rows.append(subset.iloc[0])
if model_rows:
    model_rows_df = pd.DataFrame(model_rows)
    main_out = pd.concat([human_rows, model_rows_df], ignore_index=True)
else:
    main_out = human_rows.copy()
main_out = main_out[main_columns]

main_out.to_csv(OUT_DIR / "main_joint_table.csv", index=False)

latex_lines = [
    r"\begin{table*}[t]",
    r"\centering",
    r"\footnotesize",
    r"\caption{Summary of representative human and model results. Full Hugging Face repository paths are listed in Tables~\ref{tab:models-mlm}--\ref{tab:models-s2s}; shortened model names are used here for readability. A checkmark indicates that the model passed the corresponding test ($p > 0.05$). Models were selected to represent four outcome profiles: passing both thresholds, passing only the formal threshold, passing only the functional threshold, and failing both.}",
    r"\label{tab:summary-results}",
    r"\begin{adjustbox}{max width=\textwidth}",
    r"\begin{tabular}{llccccccc}",
    r"\toprule",
    r"Model & Arch. & Bias & Empirical (SD) & Predicted (SD) & TPR (SD) & DxN test & TPR test & Accuracy \\",
    r"\midrule",
]

for _, row in main_out.iterrows():
    latex_lines.append(
        f"{row['Model']} & {row['Arch.']} & {row['Bias']} & {row['Empirical (SD)']} & {row['Predicted (SD)']} & {row['TPR (SD)']} & {row['DxN test']} & {row['TPR test']} & {row['Accuracy']} \\\\"
    )

latex_lines.extend([
    r"\bottomrule",
    r"\end{tabular}",
    r"\end{adjustbox}",
    r"\end{table*}",
])

(OUT_DIR / "main_joint_table.tex").write_text("\n".join(latex_lines), encoding="utf-8")

# Write CSV with LaTeX-formatted headers
csv_header = r"Model,Arch.,\\texttt{bias},\\texttt{empirical} (SD),\\texttt{predicted} (SD),TPR (SD),DxN test,TPR test,Accuracy"
csv_content = csv_header + "\n"
for _, row in appendix_df.iterrows():
    csv_content += f"{row['Model']},{row['Arch.']},{row['Bias']},{row['Empirical (SD)']},{row['Predicted (SD)']},{row['TPR (SD)']},{row['DxN test']},{row['TPR test']},{row['Accuracy']}\n"

(full_table_outdir.with_suffix(".csv")).write_text(csv_content, encoding="utf-8")

# Write TeX file with table environment
tex_lines = [
    r"\begin{table*}[]",
    r"\begin{adjustbox}{max width=\textwidth}",
    r"\begin{tabular}{llccccccc}",
    r"\toprule",
    r"Model & Arch. & \texttt{bias} & \texttt{empirical} (SD) & \texttt{predicted} (SD) & TPR (SD) & \texttt{D$\times$N} test & TPR test & Accuracy \\",
    r"\midrule",
]

# Add human rows
for _, row in appendix_df.iloc[:2].iterrows():
    tex_lines.append(f"{row['Model']} & {row['Arch.']} & {row['Bias']} & {row['Empirical (SD)']} & {row['Predicted (SD)']} & {row['TPR (SD)']} & {row['DxN test']} & {row['TPR test']} & {row['Accuracy']} \\\\")

tex_lines.append(r"\midrule")

# Add model rows
for _, row in appendix_df.iloc[2:].iterrows():
    tex_lines.append(f"{row['Model']} & {row['Arch.']} & {row['Bias']} & {row['Empirical (SD)']} & {row['Predicted (SD)']} & {row['TPR (SD)']} & {row['DxN test']} & {row['TPR test']} & {row['Accuracy']} \\\\")

tex_lines.extend([
    r"\bottomrule",
    r"\end{tabular}",
    r"\end{adjustbox}",
    r"\end{table*}",
])

(full_table_outdir.with_suffix(".tex")).write_text("\n".join(tex_lines), encoding="utf-8")

print(f"Generated appendix overlap table with {len(appendix_df)} rows (2 human + {len(appendix_df)-2} models)")
print(f"  CSV: {OUT_DIR / 'appendix_full_results_table.csv'}")
print(f"  TeX: {OUT_DIR / 'appendix_full_results_table.tex'}")
print("Rewrote main_joint_table to paper format at", OUT_DIR / "main_joint_table.csv")
print("Rewrote main_joint_table.tex with table* / adjustbox wrapper")


Generated appendix overlap table with 51 rows (2 human + 49 models)
  CSV: figures/appendix_full_results_table.csv
  TeX: figures/appendix_full_results_table.tex
Rewrote main_joint_table to paper format at figures/main_joint_table.csv
Rewrote main_joint_table.tex with table* / adjustbox wrapper


# Tables and Figures for paper

In [15]:
# Delete stale paper outputs before regenerating artifacts.
allowed_outputs = {
    "main_joint_table.csv",
    "main_joint_table.tex",
    "human_baseline_overlap.tex",
    "main_three_facet_figure.png",
    "appendix_full_results_table.csv",
    "appendix_full_results_table.tex",
    "appendix_accuracy_figure.png",
    "appendix_full_tpr_figure.png",
    "appendix_full_overlap_human_figure.png",
    "appendix_full_overlap_models_figure.png",
    # Cross-step artifacts produced by earlier pipeline notebooks (preserve, do not delete):
    "human_tpr_vs_window.png",                 # human_baseline.ipynb §6
    "human_antecedent_distance_hist.png",      # human_baseline.ipynb §6
    "appendix_full_results_table_corrected.tex",  # lm_tpr_analysis.ipynb §4
    "appendix_full_tpr_figure_corrected.png",     # lm_tpr_analysis.ipynb §5
    "appendix_full_tpr_vs_window_models_figure.png",  # lm_tpr_analysis.ipynb §7
}
for path in OUT_DIR.iterdir():
    if path.is_file() and path.name not in allowed_outputs:
        path.unlink()

print(f"Cleaned {OUT_DIR} and preserved current paper artifact filenames.")


Cleaned figures and preserved current paper artifact filenames.


In [16]:
# Figures for the paper.


# Main three-facet figure with shared model colors and architecture shapes.
selected_overlap = selected[["model", "arch", "mean_pred_overlap", "mean_emp_overlap"]].copy()
selected_overlap["kind"] = "Model"


child_emp = np.nan
child_pred = np.nan
mother_emp = np.nan
mother_pred = np.nan
if "human_baseline" in globals() and isinstance(human_baseline, pd.DataFrame):
    hb = human_baseline.copy()
    hb["speaker_norm"] = hb["speaker"].astype(str).str.lower().str.strip()
    child_mask = ~hb["speaker_norm"].str.contains("_mot", na=False)
    mother_mask = hb["speaker_norm"].str.contains("_mot", na=False)
    child_pred = hb.loc[child_mask, "naive_zipf_predicted"].mean()
    child_emp = hb.loc[child_mask, "empirical"].mean()
    mother_pred = hb.loc[mother_mask, "naive_zipf_predicted"].mean()
    mother_emp = hb.loc[mother_mask, "empirical"].mean()

human_ref = pd.DataFrame([
    {"model": "Child (Human)", "arch": "Human", "mean_pred_overlap": child_pred, "mean_emp_overlap": child_emp, "kind": "Human"},
    {"model": "Caretaker (Human)", "arch": "Human", "mean_pred_overlap": mother_pred, "mean_emp_overlap": mother_emp, "kind": "Human"},
])
human_ref = human_ref[human_ref[["mean_pred_overlap", "mean_emp_overlap"]].notna().all(axis=1)].copy()
overlap_facet = pd.concat([selected_overlap, human_ref], ignore_index=True)

model_order = [spec.get("model") for spec in MAIN_MODEL_SPECS]
model_order = [m for m in model_order if m in set(selected["model"].astype(str).tolist())]
if not model_order:
    model_order = selected["model"].astype(str).tolist()
order_map = {name: idx for idx, name in enumerate(model_order)}

# Build TPR plot rows without humans so model spacing stays consistent.
selected_tpr_plot = selected[["model", "arch", "mean_tpr", "sd_tpr", "TPR_pass"]].copy()
selected_tpr_plot["_ord"] = selected_tpr_plot["model"].map(order_map)
selected_tpr_plot = selected_tpr_plot.sort_values("_ord", ascending=False).reset_index(drop=True)

selected_acc_plot = selected[["model", "arch", "det_accuracy", "DxN_pass"]].copy()
selected_acc_plot["_ord"] = selected_acc_plot["model"].map(order_map)
selected_acc_plot = selected_acc_plot.sort_values("_ord", ascending=False).reset_index(drop=True)

# Align display order for accuracy and TPR facets.
display_order = selected_tpr_plot["model"].tolist()
selected_tpr_plot["display_model"] = _unique_plain_labels(display_order)
selected_tpr_plot["ypos"] = np.arange(len(display_order))

selected_acc_plot = selected_acc_plot.set_index("model").reindex(display_order).reset_index()
selected_acc_plot["display_model"] = selected_tpr_plot["display_model"]
selected_acc_plot["ypos"] = selected_tpr_plot["ypos"]

model_palette = sns.color_palette("tab10", n_colors=max(len(model_order), 3))
model_colors = {name: model_palette[idx % len(model_palette)] for idx, name in enumerate(model_order)}

fig = plt.figure(figsize=(19.5, max(6.2, 0.5 * len(selected_tpr_plot) + 2.5)))
gs = fig.add_gridspec(1, 3, width_ratios=[1.35, 1.0, 1.0], wspace=0.14)
ax_overlap = fig.add_subplot(gs[0, 0])
ax_acc = fig.add_subplot(gs[0, 1])
ax_tpr = fig.add_subplot(gs[0, 2], sharey=ax_acc)

# Facet 1: overlap (no text labels; identity via model color).
for _, row in overlap_facet.iterrows():
    if row["kind"] == "Human":
        human_color = child_color if row["model"] == "Child (Human)" else caretaker_color
        ax_overlap.scatter(row["mean_pred_overlap"], row["mean_emp_overlap"], marker="D", s=95, color=human_color, edgecolors="black", linewidths=0.6, zorder=4)
    else:
        ax_overlap.scatter(row["mean_pred_overlap"], row["mean_emp_overlap"], marker=arch_markers.get(row["arch"], "o"), s=85, color=model_colors.get(row["model"], "gray"), edgecolors="black", linewidths=0.6, zorder=3)
ax_overlap.plot([0.0, 0.5], [0.0, 0.5], linestyle="--", color="#ff9a9a", linewidth=1.0)
ax_overlap.set_xlim(0.1, 0.4)
ax_overlap.set_ylim(0.1, 0.4)
ax_overlap.set_xlabel("Mean predicted overlap")
ax_overlap.set_ylabel("Mean empirical overlap")
ax_overlap.set_title("Determiner-noun (DxN) overlap")

# Facet 2: accuracy.
for _, row in selected_acc_plot.iterrows():
    if pd.notna(row.get("det_accuracy")) and bool(row.get("DxN_pass", False)):
        ax_acc.axhspan(row["ypos"] - 0.45, row["ypos"] + 0.45, color="#d9f2d9", alpha=0.6, zorder=0)
for _, row in selected_acc_plot.iterrows():
    if pd.notna(row.get("det_accuracy")):
        ax_acc.scatter(row["det_accuracy"], row["ypos"], s=55, marker=arch_markers.get(row["arch"], "o"), color=model_colors.get(row["model"], "gray"), edgecolors="black", linewidths=0.5, zorder=3)
ax_acc.axvline(MAJORITY_DET_BASELINE, color="gray", linestyle="--", linewidth=1.2)
ax_acc.legend(handles=[Line2D([0], [0], color="gray", linestyle="--", linewidth=1.2, label="Majority determiner"), Line2D([0], [0], color="#d9f2d9", linewidth=8, label="DxN test passed")], loc="upper right", fontsize=7, frameon=True, borderpad=0.3, handlelength=2.0)
ax_acc.set_xlim(0, 1)
ax_acc.set_yticks(selected_acc_plot["ypos"].tolist())
#ax_acc.set_yticklabels(selected_acc_plot["display_model"].tolist())
#ax_acc.tick_params(axis="y", direction="in", pad=2, labelleft=True, labelright=False)
for tick in ax_acc.get_yticklabels():
    tick.set_horizontalalignment("left")
    tick.set_x(0.02)
ax_acc.set_xlabel("Determiner accuracy")
ax_acc.set_ylabel("")
ax_acc.set_title("Determiner prediction accuracy")

# Facet 3: TPR.
for _, row in selected_tpr_plot.iterrows():
    if bool(row["TPR_pass"]):
        ax_tpr.axhspan(row["ypos"] - 0.45, row["ypos"] + 0.45, color="#d9f2d9", alpha=0.6, zorder=0)
for _, row in selected_tpr_plot.iterrows():
    ax_tpr.errorbar(x=row["mean_tpr"], y=row["ypos"], xerr=(row["sd_tpr"] if pd.notna(row["sd_tpr"]) else 0.0), fmt=arch_markers.get(row["arch"], "o"), color=model_colors.get(row["model"], "gray"), ecolor=model_colors.get(row["model"], "gray"), capsize=3, markersize=6, markeredgecolor="black", markeredgewidth=0.5, zorder=3)

ax_tpr.axvline(ADULT_TPR_POPULATION_MEAN, linestyle="--", color="black", linewidth=1.2)
ax_tpr.legend(handles=[Line2D([0], [0], color="black", linestyle="--", linewidth=1.2, label="Other-adult population mean"), Line2D([0], [0], color="#d9f2d9", linewidth=8, label="TPR test passed")], loc="upper right", fontsize=7, frameon=True, borderpad=0.3, handlelength=2.0)
ax_tpr.set_xlim(0, 1)
ax_tpr.tick_params(axis="y", left=False, labelleft=False, labelright=False)
ax_tpr.set_xlabel("Mean model TPR")
ax_tpr.set_ylabel("")
ax_tpr.set_title("Transitional probability of reference (TPR)")

model_arch_map = dict(zip(selected["model"], selected["arch"]))
model_handles = [Line2D([0], [0], marker=arch_markers.get(model_arch_map.get(model_name), "o"), color="w", label=model_name, markerfacecolor=model_colors[model_name], markeredgecolor="black", markersize=7, linewidth=0) for model_name in model_order]
shared_legend = [Line2D([0], [0], marker="D", color="w", label="Children's mean overlap", markerfacecolor=child_color, markeredgecolor="black", markersize=8), Line2D([0], [0], marker="D", color="w", label="Caretaker's mean overlap", markerfacecolor=caretaker_color, markeredgecolor="black", markersize=8)] +       model_handles
fig.legend(handles=shared_legend, loc="lower center", ncol=4, frameon=True, bbox_to_anchor=(0.5, 0.1))
fig.suptitle("Formal & Functional benchmark of Determiner-Noun competence", y=0.98, fontsize=13)
fig.subplots_adjust(bottom=0.30, top=0.88, wspace=0.14)
fig.savefig(OUT_DIR / "main_three_facet_figure.png", dpi=300, bbox_inches="tight")
plt.close(fig)

# Appendix accuracy figure: majority baseline line and pass shading.
acc_fig = acc_summary_use.merge(overlap_summary_use[["model", "model_type", "arch", "DxN_pass"]], on=["model", "model_type"], how="left").sort_values("det_accuracy", ascending=False).reset_index(drop=True)
acc_fig["display_model"] = _unique_plain_labels(acc_fig["model"].tolist(), acc_fig["arch"].tolist())
acc_fig["ypos"] = np.arange(len(acc_fig))

fig, ax = plt.subplots(figsize=(8.2, max(6, 0.25 * len(acc_fig) + 1.5)))
for _, row in acc_fig.iterrows():
    if bool(row.get("DxN_pass", False)):
        ax.axhspan(row["ypos"] - 0.45, row["ypos"] + 0.45, color="#d9f2d9", alpha=0.6, zorder=0)
for _, row in acc_fig.iterrows():
    ax.scatter(row["det_accuracy"], row["ypos"], s=45, color=arch_colors.get(row["arch"], "gray"), alpha=0.9, zorder=3)
ax.axvline(MAJORITY_DET_BASELINE, color="gray", linestyle="--", linewidth=1.5, label="Majority determiner")
ax.set_xlim(0, 1)
ax.set_yticks(acc_fig["ypos"].tolist())
ax.set_yticklabels(acc_fig["display_model"].tolist())
ax.set_xlabel("Determiner accuracy (child, probabilistic)")
ax.set_ylabel("Model")
ax.set_title("Appendix: full determiner accuracy")
ax.legend(handles=[
    Line2D([0], [0], marker="o", color="w", label=arch, markerfacecolor=color, markeredgecolor="black", markersize=8)
    for arch, color in arch_colors.items()
 ] + [
    Line2D([0], [0], color="gray", linestyle="--", linewidth=1.5, label="Majority determiner"),
    Line2D([0], [0], color="#d9f2d9", linewidth=8, label="passes DxN overlap"),
], title="Legend", loc="upper left", bbox_to_anchor=(0.01, 0.99), frameon=True)
fig.tight_layout()
fig.savefig(OUT_DIR / "appendix_accuracy_figure.png", dpi=300)
plt.close(fig)

print("Saved figure artifacts to", OUT_DIR)
print("- main_three_facet_figure.png")
print("- appendix_accuracy_figure.png")


Saved figure artifacts to figures
- main_three_facet_figure.png
- appendix_accuracy_figure.png
